## Face-to-Voice Dataset Creation

### Overview
This notebook processes the **VGGFace2** and **VoxCeleb2** datasets to create a training dataset for the Face-to-Voice model.  
The goal is to map **Facial Features** to **Voice Style Vectors** for each speaker.

### Workflow
1.  **Setup**: Import libraries, define constants, and load StyleTTS2 model.
2.  **Face Selection**: Select the best-quality face image for each speaker from VGGFace2 based on detection confidence.
3.  **Face Processing**: Extract ArcFace embeddings and predict age from the selected face images.
4.  **Audio Processing**: Extract StyleTTS2 style vectors from VoxCeleb2 audio files for each speaker.
5.  **Merging & Output**: Combine face vectors, style vectors, age, and gender metadata into a single dataset and save as pickle.

### Dataset Preparation  
**Audio Data**  
Source: [Hugging Face (Reverb/voxceleb2)](https://huggingface.co/datasets/Reverb/voxceleb2)  
Instructions:  
    1. Download the split archive files (`aac.7z.001`, etc.), combine, and extract them.  
    2. Place the contents in the `data/voxceleb2` directory.

**Face Image Data**  
Source: [Academic Torrents (VGGFace2)](https://academictorrents.com/details/535113b8395832f09121bc53ac85d7bc8ef6fa5b)  
Instructions:  
    1. Download the dataset via a Torrent client.  
    2. Place the contents in the `data/vggface2` directory.

**Important:** Use speakers that appear in **both** the Audio and Face datasets according to the metadata CSV. Any unmatched entries are excluded.

### 1. Setup

In [ ]:
import os
import shutil
import contextlib
import warnings
import subprocess
from glob import glob

import numpy as np
import pandas as pd
import librosa
import torch
from tqdm import tqdm
from moviepy.config import FFMPEG_BINARY
from tensorflow.keras import backend as K

# AI Models
from deepface import DeepFace
from styletts2.tts import StyleTTS2, preprocess

# Settings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# --- Configuration ---
DATASET_ROOT = '../data' 
FACE_ROOT = os.path.normpath(os.path.join(DATASET_ROOT, 'vggface2')) 
VOICE_ROOT = os.path.normpath(os.path.join(DATASET_ROOT, 'voxceleb2')) 
META_CSV = os.path.normpath(os.path.join(VOICE_ROOT, 'vox2_meta.csv'))

# Directories
FACE_DIR = os.path.normpath(os.path.join(FACE_ROOT, 'images'))
BEST_FACE_DIR = os.path.normpath(os.path.join(FACE_ROOT, 'images_best'))
AUDIO_DIR = os.path.normpath(os.path.join(VOICE_ROOT, 'aac'))

# Output Path
AUDIO_PKL = os.path.normpath(os.path.join(DATASET_ROOT, 'audio_vectors.pkl'))
FACE_PKL = os.path.normpath(os.path.join(DATASET_ROOT, 'face_vectors.pkl'))
OUTPUT_PKL = os.path.normpath(os.path.join(DATASET_ROOT, 'face_to_voice_dataset.pkl'))

# Settings for face detection
BACKEND = 'retinaface' 
TARGET_CONFIDENCE = 0.99

# Audio Settings for StyleTTS 2
SAMPLE_RATE = 24000
CHANNELS = 1  # Mono

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Load Models ---
with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
    tts = StyleTTS2()

### 2. Face Selection

In [ ]:
if not os.path.exists(BEST_FACE_DIR):
    os.makedirs(BEST_FACE_DIR)

# Get speaker folders
speaker_folders = glob(os.path.join(FACE_DIR, 'n*'))

for folder in tqdm(speaker_folders, desc='Selecting Best Faces'):
    speaker_id = os.path.basename(folder)
    dst_path = os.path.join(BEST_FACE_DIR, f'{speaker_id}.jpg')
    
    # Skip if already processed
    if os.path.exists(dst_path):
        continue

    # Get images
    images = glob(os.path.join(folder, "*.jpg"))
    if not images:
        continue

    best_score = -1
    best_img_path = None

    for img_path in images:
        try:
            # Detect faces
            face_objs = DeepFace.extract_faces(
                img_path=img_path, 
                detector_backend=BACKEND, 
                enforce_detection=True,
                align=False
            )
            
            # If multiple faces are detected, pick the one with highest confidence
            current_face = max(face_objs, key=lambda x: x['confidence'])
            score = current_face['confidence']

            # Update best score
            if score > best_score:
                best_score = score
                best_img_path = img_path

            # Early Stopping: If perfect face is found, stop checking other images for this person
            if best_score >= TARGET_CONFIDENCE:
                break

        except:
            # If DeepFace cannot detect a face, skip this image
            continue

    # Save the best image found
    if best_img_path:
        shutil.copy(best_img_path, dst_path)

### 3. Face Processing

In [ ]:
data_dict = {}
face_images = glob(os.path.join(BEST_FACE_DIR, '*.jpg'))

# Load existing data if available
if os.path.exists(FACE_PKL):
    try:
        df_existing = pd.read_pickle(FACE_PKL)
        data_dict = df_existing.to_dict('index')
        print(f'Loaded {len(data_dict)} existing records from {FACE_PKL}')
    except:
        print('Starting fresh...')

processed_ids = set(data_dict.keys())

for i, img_path in enumerate(tqdm(face_images, desc='Processing')):
    vec = None
    
    try:
        # Get VGG ID
        vgg_id = os.path.splitext(os.path.basename(img_path))[0]
    except:
        continue
    
    # Skip if already processed
    if vgg_id in processed_ids:
        continue
    
    try:
        # Extract face
        face_objs = DeepFace.extract_faces(
                    img_path=img_path, 
                    detector_backend=BACKEND,
                    enforce_detection=True,
                    normalize_face=False
        )   
        best_face = max(face_objs, key=lambda x: x['confidence'])
        face_img = best_face['face']

        # Get face vector
        embedding_objs = DeepFace.represent(
            img_path=face_img,
            model_name='ArcFace',
            detector_backend='skip',
            align=False
        )
        vector = embedding_objs[0]['embedding']

        # Get Age
        demographics = DeepFace.analyze(
            img_path=face_img,
            actions=['age'],
            detector_backend='skip',
            align=False,
            silent=True
        )
        age = demographics[0]['age']

        # Store Data
        data_dict[vgg_id] = {
            'face_vector': np.array(vector, dtype=np.float32),
            'age': age
        }        

    except Exception as e:
        print(f'Error processing {vgg_id}: {e}')
        continue
    
    # Save every 300 iterations
    if (i + 1) % 300 == 0:
        df_temp = pd.DataFrame.from_dict(data_dict, orient='index')
        df_temp.index.name = 'vgg_id'
        df_temp.to_pickle(FACE_PKL)
        print(f'Saved {len(df_temp)} records at iteration {i + 1}')

# Final save
df_face = pd.DataFrame.from_dict(data_dict, orient='index')
df_face.index.name = 'vgg_id'
df_face.to_pickle(FACE_PKL)

print(f'Total Faces: {len(df_face)}')
print(df_face.head())
print(f'Final save to {FACE_PKL}')

### 4. Audio Processing

In [ ]:
def load_m4a(path):
    # Decode m4a to wav
    try:
        cmd = [
            FFMPEG_BINARY,
            '-i', path,
            '-f', 'f32le',
            '-ac', '1', 
            '-ar', str(SAMPLE_RATE),
            '-acodec', 'pcm_f32le', 
            '-vn', '-loglevel',
            'quiet', '-'
        ]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        out, _ = proc.communicate()
        wav = np.frombuffer(out, np.float32)
        return wav
    except:
        return None

In [ ]:
data_dict = {}
speaker_folders = glob(os.path.join(AUDIO_DIR, 'id*'))

# Load existing data if available
if os.path.exists(AUDIO_PKL):
    try:
        df_existing = pd.read_pickle(AUDIO_PKL)
        data_dict = df_existing['style_vectors'].to_dict()
        print(f'Loaded {len(data_dict)} existing records from {AUDIO_PKL}')
    except:
        print('Starting fresh...')

processed_ids = set(data_dict.keys())

for i, folder in enumerate(tqdm(speaker_folders, desc='Processing')):
    vec = None
    
    try:
        # Get Vox ID
        vox_id = os.path.basename(folder)
    except: 
        continue

    # Skip if already processed
    if vox_id in processed_ids:
        continue

    m4a_files = glob(os.path.join(folder, '**', '*.m4a'), recursive=True)
    if not m4a_files:
        continue

    best_audio_path = max(m4a_files, key=os.path.getsize)

    # Load Audio
    wav = load_m4a(best_audio_path)
    if wav is None or wav.size == 0:
        continue

    # Style Extraction (Styletts2 compute_style())
    try:
        audio, _ = librosa.effects.trim(wav, top_db=30)
        mel_tensor = preprocess(audio).to(DEVICE)

        with torch.no_grad():
            ref_s = tts.model.style_encoder(mel_tensor.unsqueeze(1))
            ref_p = tts.model.predictor_encoder(mel_tensor.unsqueeze(1))

        style_vec = torch.cat([ref_s, ref_p], dim=1)

        # Save to list (Convert to compact numpy)
        vec = style_vec.squeeze().cpu().numpy().astype(np.float32)

    except Exception as e:
        print(f'Skipping ID({vox_id}): {e}')
        pass

    # Only add speaker if they have vectors
    if vec is not None or vec.size > 0:
        data_dict[vox_id] = vec
    
    # Save every 300 iterations
    if (i + 1) % 300 == 0:
        df_temp = pd.Series(data_dict).to_frame(name='style_vectors')
        df_temp.index.name = 'vox_id'
        df_temp.to_pickle(AUDIO_PKL)
        print(f'Saved {len(df_temp)} records at iteration {i + 1}')

# Convert to DataFrame
df_voice = pd.Series(data_dict).to_frame(name='style_vectors')
df_voice.index.name = 'vox_id'

print(f'Total Speakers: {len(df_voice)}')
print(df_voice.head())

# Save to Pickle
df_voice.to_pickle(AUDIO_PKL)
print(f'Saved to {AUDIO_PKL}')

### 5. Merging & Output

In [ ]:
# --- Load Metadata ---
df_meta = pd.read_csv(META_CSV)

# Clean column names
df_meta.columns = df_meta.columns.str.strip()

# Clean values
df_meta['VoxCeleb2 ID'] = df_meta['VoxCeleb2 ID'].str.strip()
df_meta['VGGFace2 ID'] = df_meta['VGGFace2 ID'].str.strip()
df_meta['Gender'] = df_meta['Gender'].str.strip()

# --- Load Face Vectors ---
if os.path.exists(FACE_PKL):
    df_face = pd.read_pickle(FACE_PKL)
    print(f'Loaded face vectors: {len(df_face)}')
else:
    raise FileNotFoundError(f'Face PKL not found: {FACE_PKL}')

# --- Load Audio Vectors ---
if os.path.exists(AUDIO_PKL):
    df_audio = pd.read_pickle(AUDIO_PKL)
    print(f'Loaded audio vectors: {len(df_audio)}')
else:
    raise FileNotFoundError(f'Audio PKL not found: {AUDIO_PKL}')

# --- Merge ---
# Start with Metadata
df_final = df_meta.copy()

# Merge with Face Data
df_final = df_final.merge(
    df_face,
    left_on='VGGFace2 ID',
    right_index=True,
    how='inner'
)

# Merge with Voice Data
df_final = df_final.merge(
    df_audio,
    left_on='VoxCeleb2 ID',
    right_index=True,
    how='inner'
)

# Reorder columns
target_cols = ['VoxCeleb2 ID', 'VGGFace2 ID', 'Gender', 'age', 'face_vector', 'style_vectors']
df_final = df_final[target_cols]

# Drop rows with missing values
df_final = df_final.dropna()

print(f'Final Data Count: {len(df_final)}')
print(df_final.head())

# Save
df_final.to_pickle(OUTPUT_PKL)
print(f'Successfully saved to {OUTPUT_PKL}')